In [1]:
import torch
import numpy as np
from IPython.display import display, Image
from matplotlib import pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seq_len = 500
canvas_dim = 32
def moving_bar():
    bar_width_max = 10
    bar_width_min = 2
    canvas = np.zeros((seq_len, 3, canvas_dim, canvas_dim))
    color_pick = np.random.randint(0, 2)
    class_i = 2
    bar_x_pos = np.random.randint(0, canvas_dim)
    bar_width = np.random.randint(bar_width_min, bar_width_max)
    disp = np.random.randint(1, 8)
    orientation = np.random.randint(0, 2)
    for t in range(seq_len):
        bar_on = bar_x_pos%canvas_dim
        bar_off = (bar_x_pos+bar_width)%canvas_dim
        if bar_on > bar_off:
            swap = bar_on
            bar_off = bar_on
            bar_on = swap
        if orientation:
            canvas[t, color_pick, bar_on:bar_off, :] = np.ones((bar_off-bar_on, canvas_dim))
        else:
            canvas[t, color_pick, :, bar_on:bar_off] = np.ones((bar_off-bar_on, canvas_dim)).T
        bar_x_pos += disp
    return canvas, class_i*orientation + color_pick

In [2]:
def return_datasamples(num_samples):
    X_train = []
    Y_train = np.zeros((num_samples, seq_len, 4))
    ramp = np.linspace(0, 2, seq_len)
    for s in range(num_samples):
        v, c = moving_bar()
        X_train.append(v)
        Y_train[s, :, c] = ramp

    X_train = torch.from_numpy(np.array(X_train, dtype='float32'))
    Y_train = torch.from_numpy(np.array(Y_train, dtype='float32'))

    return(torch.permute(X_train, [1, 0, 2, 3, 4]).to(device),
           torch.permute(Y_train, [1, 0, 2]).to(device))

Xt, Yt = return_datasamples(100)

np.save('X.npy', Xt.cpu().numpy())
np.save('Y.npy', Yt.cpu().numpy())